# Определение возраста пользователей на основе их цифрового поведения

## Цель проекта

Построить модель многоклассовой классификации, которая по данным о цифровом поведении
анонимного пользователя будет предсказывать его возрастную категорию.

Модель предназначена для:
- таргетированной рекламы по возрастным сегментам;
- повышения эффективности маркетинговых кампаний;
- снижения рисков показа контента для взрослых несовершеннолетним пользователям.

## Описание проекта

Данные представлены несколькими CSV-файлами, полученными из различных источников.
Каждый набор данных содержит информацию о действиях пользователя в цифровой среде.

В рамках проекта предполагается:
1. Проанализировать каждый датафрейм.
2. Объединить данные в единую таблицу.
3. Выполнить предобработку признаков:
   - обработку пропусков;
   - кодирование категориальных признаков;
   - масштабирование числовых признаков.
4. Построить базовую модель многоклассовой классификации.
5. Оценить качество модели с использованием релевантных метрик.
6. Сравнить базовый классификатор с более сложными моделями.

## Подготовка среды и библиотек

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

import joblib

import warnings
warnings.filterwarnings("ignore")

sns.set(style="whitegrid")

## Исследовательский анализ данных

In [ ]:
users = pd.read_csv("/datasets/ds_s13_users.csv")
visits = pd.read_csv("/datasets/ds_s13_visits.csv")
ads = pd.read_csv("/datasets/ads_activity.csv")
surf = pd.read_csv("/datasets/surf_depth.csv")
device = pd.read_csv("/datasets/primary_device.csv")
cloud = pd.read_csv("/datasets/cloud_usage.csv")

In [ ]:
print("users:", users.shape)
print("visits:", visits.shape)
print("ads:", ads.shape)
print("surf:", surf.shape)
print("device:", device.shape)
print("cloud:", cloud.shape)

После загрузки файлов получены следующие размеры таблиц:

- `users` - (5913, 2). Содержит целевую переменную `age_category` и идентификатор пользователя `user_id`.
- `visits` - (1065745, 5). Лог посещений сайтов, основной источник поведенческих признаков.
- `ads_activity` - (5826, 2). Категориальный признак, отражающий частоту взаимодействия с рекламой (CTR).
- `surf_depth` - (5715, 2). Категориальный признак глубины просмотра сайтов в рамках сессии.
- `primary_device` - (5669, 2). Тип основного устройства пользователя.
- `cloud_usage` - (5680, 2). Булевый признак использования облачных технологий.

Число записей в дополнительных таблицах меньше, чем в `users`, поэтому при объединении данных по `user_id` у части пользователей будут отсутствующие значения (NaN). Эти пропуски будут обработаны на этапе предобработки данных.

In [ ]:
users["age_category"].value_counts(normalize=True)

Распределение классов `age_category` показывает умеренный дисбаланс:

- класс 4 (56+) - ~30%
- класс 2 (26-40) - ~25%
- класс 3 (41-55) - ~21%
- класс 0 (младше 18) - ~15%
- класс 1 (18-25) - ~9%

Сильного перекоса классов нет, однако класс 1 представлен заметно реже остальных. Поэтому, помимо Accuracy, при оценке моделей будет использоваться метрика F1 (macro), которая чувствительна к качеству предсказаний на редких классах.

In [ ]:
visits_count = (
    visits
    .groupby("user_id")
    .size()
    .reset_index(name="visits_count")
)

visits_count.head()

In [ ]:
daytime_counts = (
    visits
    .groupby(["user_id", "daytime"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

daytime_counts.head()

In [ ]:
unique_sites = (
    visits
    .groupby("user_id")["website_category"]
    .nunique()
    .reset_index(name="unique_website_categories")
)

unique_sites.head()

Таблица `visits` содержит более 1 млн записей и отражает поведение пользователей в цифровой среде.  
Так как итоговая модель должна работать на уровне пользователя, данные были агрегированы по `user_id`.

Для каждого пользователя рассчитано общее число посещений сайтов (`visits_count`).  
Этот признак отражает общую активность пользователя в интернете.

Для каждого пользователя подсчитано количество визитов в разрезе категорий `daytime`:

- утро  
- день  
- вечер  
- ночь  

Данные признаки позволяют учитывать поведенческие особенности пользователей в зависимости от времени суток.

Рассчитано количество уникальных категорий сайтов (`unique_website_categories`), посещённых пользователем.  

Этот признак характеризует широту интересов и цифровую активность пользователя.

В результате лог посещений был преобразован из событийного формата (1 строка = 1 посещение) в агрегированный формат (1 строка = 1 пользователь), пригодный для последующего объединения с остальными таблицами.

In [ ]:
df = users.copy()

df = df.merge(visits_count, on="user_id", how="left")
df = df.merge(daytime_counts, on="user_id", how="left")
df = df.merge(unique_sites, on="user_id", how="left")
df = df.merge(ads, on="user_id", how="left")
df = df.merge(surf, on="user_id", how="left")
df = df.merge(device, on="user_id", how="left")
df = df.merge(cloud, on="user_id", how="left")

После формирования агрегированных поведенческих признаков таблицы были объединены по ключу `user_id`.

В качестве основной таблицы использована `users`, содержащая целевую переменную `age_category`.  
К ней последовательно присоединены:

- агрегированные признаки из `visits`;
- данные о рекламной активности (`ads_activity`);
- глубина просмотра сайтов (`surf_depth`);
- тип основного устройства (`primary_device`);
- использование облачных сервисов (`cloud_usage`).

Объединение выполнялось по схеме `left join`, что позволило сохранить всех пользователей из основной таблицы, даже если по отдельным признакам информация отсутствует.

В результате сформирован единый датафрейм, содержащий:

- числовые поведенческие признаки;
- категориальные характеристики пользователя;
- целевую переменную `age_category`.

Полученный набор данных будет использоваться для дальнейшей обработки пропусков и построения модели.

In [ ]:
df.shape

In [ ]:
df.info()

После объединения таблиц размер итогового датафрейма составил **(6146, 12)**.

Количество строк превышает исходное число пользователей в таблице `users` (5913).  
Это указывает на наличие повторяющихся `user_id` в одной или нескольких присоединённых таблицах.

Таким образом, перед переходом к этапу предобработки необходимо:

- проверить наличие дубликатов `user_id`;
- убедиться, что в итоговом датасете каждой строке соответствует один уникальный пользователь;
- при необходимости удалить повторяющиеся записи.

В датасете представлены:

- 7 числовых признаков (`int64`);
- 5 категориальных признаков (`object`).

Числовые признаки сформированы на основе агрегированных поведенческих данных.  
Категориальные признаки отражают характеристики пользователя (рекламная активность, глубина просмотра, устройство,bиспользование облака).

Пропуски присутствуют в следующих признаках:

- `ads_activity`
- `surf_depth`
- `primary_device`
- `cloud_usage`

Пропуски возникли из-за отсутствия информации по отдельным пользователям в соответствующих таблицах.

На этапе предобработки будет принято решение о корректной обработке этих пропусков.

In [ ]:
df["user_id"].nunique(), df.shape[0]

In [ ]:
print("ads:", ads["user_id"].duplicated().sum())
print("surf:", surf["user_id"].duplicated().sum())
print("device:", device["user_id"].duplicated().sum())
print("cloud:", cloud["user_id"].duplicated().sum())

In [ ]:
ads = ads.drop_duplicates(subset="user_id")
surf = surf.drop_duplicates(subset="user_id")
device = device.drop_duplicates(subset="user_id")
cloud = cloud.drop_duplicates(subset="user_id")

In [ ]:
df = users.copy()

df = df.merge(visits_count, on="user_id", how="left")
df = df.merge(daytime_counts, on="user_id", how="left")
df = df.merge(unique_sites, on="user_id", how="left")
df = df.merge(ads, on="user_id", how="left")
df = df.merge(surf, on="user_id", how="left")
df = df.merge(device, on="user_id", how="left")
df = df.merge(cloud, on="user_id", how="left")

df.shape

In [ ]:
df["user_id"].nunique()

In [ ]:
users.shape
users["user_id"].nunique()
users["user_id"].duplicated().sum()

In [ ]:
users = users.drop_duplicates(subset="user_id")
users.shape, users["user_id"].nunique()

In [ ]:
df = users.copy()

df = df.merge(visits_count, on="user_id", how="left")
df = df.merge(daytime_counts, on="user_id", how="left")
df = df.merge(unique_sites, on="user_id", how="left")
df = df.merge(ads, on="user_id", how="left")
df = df.merge(surf, on="user_id", how="left")
df = df.merge(device, on="user_id", how="left")
df = df.merge(cloud, on="user_id", how="left")

df.shape, df["user_id"].nunique()

In [ ]:
plt.figure()
sns.countplot(x="age_category", data=df)
plt.title("Распределение возрастных категорий")
plt.show()

При проверке ключевой таблицы `users` обнаружены дубликаты по `user_id` (87 повторов).  
Так как итоговый датасет должен содержать одну строку на пользователя, дубликаты были удалены.

После удаления дубликатов количество уникальных пользователей составило 5826.  
Далее объединение таблиц выполнялось относительно очищенной таблицы `users`, что обеспечило корректную структуру итогового датасета (1 строка = 1 пользователь).

In [ ]:
(df.isna().mean().sort_values(ascending=False) * 100).head(10)

Доля пропусков в признаках составляет:

- `ads_activity` - 3.99%
- `primary_device` - 2.69%
- `cloud_usage` - 2.51%
- `surf_depth` - 1.91%

Остальные признаки, включая целевую переменную `age_category` и все числовые поведенческие признаки, пропусков не содержат.

Пропуски наблюдаются исключительно в категориальных характеристиках пользователя.  
Вероятнее всего, это связано с отсутствием информации по отдельным пользователям в соответствующих источниках данных.

Так как доля пропусков невелика (менее 5%), удаление строк привело бы к неоправданной потере наблюдений.  
Поэтому принято решение заменить пропущенные значения отдельной категорией `"unknown"`, что позволит:

- сохранить всех пользователей в обучающей выборке;
- не искажать распределение классов;
- избежать потери информации.

In [ ]:
numeric_cols = [
    "visits_count",
    "вечер",
    "день",
    "ночь",
    "утро",
    "unique_website_categories"
]

df[numeric_cols].describe()

In [ ]:
df["total_activity"] = df[["утро","день","вечер","ночь"]].sum(axis=1)

plt.figure()
sns.histplot(df["total_activity"], bins=30)
plt.title("Распределение общей активности пользователей")
plt.show()

Статистический анализ числовых признаков показал следующее.

Среднее число визитов составляет ~183, медиана - 169.  
Максимальное значение (853) значительно превышает 75-й перцентиль (217), что указывает на наличие пользователей с аномально высокой активностью.  

Распределение имеет выраженный правый хвост, что может потребовать масштабирования признака перед обучением модели.

Наибольшая активность пользователей наблюдается днём и вечером.  
Ночная активность значительно ниже, что соответствует естественному поведенческому паттерну.

Большинство пользователей посещают от 18 до 19 различных категорий сайтов.  
Низкая вариативность признака может ограничивать его вклад в модель, однако он сохраняется для дальнейшего анализа.

In [ ]:
df[numeric_cols].corr()

In [ ]:
plt.figure()
sns.heatmap(df[numeric_cols].corr(), annot=True)
plt.title("Корреляция числовых признаков")
plt.show()

In [ ]:
df = df.drop(columns=["visits_count"])

numeric_cols = [
    "вечер",
    "день",
    "ночь",
    "утро",
    "unique_website_categories"
]

Матрица корреляции показала сильную зависимость между признаком `visits_count` и количеством визитов в разрезе времени суток (коэффициенты корреляции > 0.9).

Это объясняется тем, что `visits_count` фактически является суммой визитов по категориям времени суток.

Для предотвращения мультиколлинеарности принято решение удалить признак `visits_count`, сохранив более информативные признаки распределения активности по времени суток.

## Предобработка данных

In [ ]:
categorical_cols = ["ads_activity", "surf_depth", "primary_device", "cloud_usage"]

df[categorical_cols] = df[categorical_cols].fillna("unknown").astype(str)

Перед кодированием категориальных признаков выполнена унификация типов данных: признак `cloud_usage` содержит булевы значения (True/False), а пропуски заполняются строковым значением `"unknown"`.  
Чтобы избежать смешения типов и корректно применить One-Hot Encoding, все категориальные признаки приведены к строковому типу.

In [ ]:
numeric_cols = [
    "вечер",
    "день",
    "ночь",
    "утро",
    "unique_website_categories"
]

In [ ]:
X = df.drop(columns=["user_id", "age_category"])
y = df["age_category"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

In [ ]:
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            multi_class="multinomial",
            solver="lbfgs",
            max_iter=1000
        ))
    ]
)

В рамках предобработки были выполнены следующие шаги:

- пропущенные значения в категориальных признаках заменены на категорию "unknown";
- данные разделены на признаки и целевую переменную;
- выполнено стратифицированное разделение на обучающую и тестовую выборки;
- создан пайплайн предобработки, включающий масштабирование числовых признаков и One-Hot кодирование категориальных;
- сформирован итоговый Pipeline, объединяющий этапы предобработки и модель.


## Обучение и оценка базовой модели

In [ ]:
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification report:\n")
print(classification_report(y_test, y_pred))

Базовая многоклассовая логистическая регрессия показала Accuracy ≈ 0.58, что значительно выше случайного угадывания (≈ 0.20 для 5 классов).

Модель хорошо определяет старшую возрастную категорию (56+) и пользователей младше 18 лет.  
Однако качество предсказания для группы 18-25 лет остаётся низким (Recall ≈ 0.04), что может быть связано с:

- небольшим числом наблюдений в данном классе;
- слабой выраженностью поведенческих различий;
- отсутствием балансировки классов.

Для улучшения качества требуется дальнейшая работа с признаками и подбор гиперпараметров.

## Создание и отбор признаков

In [ ]:
model_balanced = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            multi_class="multinomial",
            solver="lbfgs",
            max_iter=1000,
            class_weight="balanced"
        ))
    ]
)

model_balanced.fit(X_train, y_train)
y_pred_bal = model_balanced.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_bal))
print(classification_report(y_test, y_pred_bal))

Для улучшения качества предсказания редкого класса (18-25 лет) была обучена модель с параметром `class_weight="balanced"`.

Сравнение с базовой моделью показало:

- Accuracy снизилась с ~0.58 до ~0.56;
- Recall для класса 18-25 вырос с 0.04 до 0.40;
- F1-score для данного класса увеличился с 0.07 до 0.31;
- Macro F1 выросла с ~0.48 до ~0.52.

Несмотря на небольшое снижение общей точности, модель стала значительно лучше распознавать редкий класс, что делает её более сбалансированной и практично применимой.

В дальнейшем в качестве основной рассматривается модель с балансировкой классов.

In [ ]:
df["total_activity"] = df[["утро", "день", "вечер", "ночь"]].sum(axis=1)

df["total_activity"] = df["total_activity"].replace(0, 1)

df["morning_ratio"] = df["утро"] / df["total_activity"]
df["day_ratio"] = df["день"] / df["total_activity"]
df["evening_ratio"] = df["вечер"] / df["total_activity"]
df["night_ratio"] = df["ночь"] / df["total_activity"]

In [ ]:
X = df.drop(columns=["user_id", "age_category"])
y = df["age_category"]

In [ ]:
numeric_cols = [
    "вечер", "день", "ночь", "утро",
    "unique_website_categories",
    "morning_ratio", "day_ratio", "evening_ratio", "night_ratio"
]
categorical_cols = ["ads_activity", "surf_depth", "primary_device", "cloud_usage"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

In [ ]:
model_balanced_ratio = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            multi_class="multinomial",
            solver="lbfgs",
            max_iter=1000,
            class_weight="balanced"
        ))
    ]
)

In [ ]:
model_balanced_ratio.fit(X_train, y_train)
y_pred = model_balanced_ratio.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Дополнительно были созданы относительные признаки (доли активности по времени суток), отражающие поведенческий профиль пользователя.

Однако добавление ratio-признаков не привело к значимому улучшению качества модели:

- Accuracy немного снизилась;
- Macro F1 осталась на прежнем уровне;
- качество распознавания редкого класса практически не изменилось.

Таким образом, наилучший баланс качества достигается моделью с параметром `class_weight="balanced"` без добавления дополнительных ratio-признаков.

В дальнейшей работе используется именно эта модель.

## Подбор гиперпараметров моделей

In [ ]:
param_grid = {
    "classifier__C": [0.01, 0.1, 1, 5, 10, 50]
}

grid = GridSearchCV(
    model_balanced,
    param_grid,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1
)

grid.fit(X_train, y_train)

In [ ]:
print("Best C:", grid.best_params_)
print("Best CV score (macro F1):", grid.best_score_)

In [ ]:
best_model = grid.best_estimator_

y_pred_best = best_model.predict(X_test)

print("Test Accuracy:", accuracy_score(y_test, y_pred_best))
print(classification_report(y_test, y_pred_best))

In [ ]:
plt.figure()
sns.heatmap(confusion_matrix(y_test, y_pred_best), annot=True, fmt="d")
plt.title("Confusion Matrix (Лучшая модель)")
plt.show()

Для оптимизации качества модели выполнен подбор коэффициента регуляризации C с использованием GridSearchCV и 5-кратной кросс-валидации.

В качестве метрики использовалась F1 (macro), поскольку задача характеризуется дисбалансом классов.

Лучший параметр:

- C = 5
- CV Macro F1 ≈ 0.51

На тестовой выборке модель показала:

- Accuracy ≈ 0.56
- Macro F1 ≈ 0.52

Подбор гиперпараметра позволил стабилизировать модель, однако значительного прироста качества по сравнению с вариантом с балансировкой классов не наблюдается.

Итоговая модель использует балансировку классов и оптимальный коэффициент регуляризации C=5.

## Подготовка артефактов модели для внедрения

In [ ]:
joblib.dump(best_model, "age_prediction_model.joblib")

In [ ]:
loaded_model = joblib.load("age_prediction_model.joblib")

sample_prediction = loaded_model.predict(X_test.iloc[:5])
print(sample_prediction)

Итоговая модель сохранена с использованием библиотеки `joblib`.

Сохранён весь Pipeline, включающий:

- предобработку данных (масштабирование и One-Hot Encoding);
- обученную модель логистической регрессии.

Это гарантирует корректную обработку новых данных при использовании модели в продакшене.

Модель может быть загружена и использована для предсказаний без повторного обучения.

## Выводы о результатах работы

В рамках проекта была построена модель многоклассовой классификации для определения возрастной категории пользователя на основе его цифрового поведения.

### Основные этапы работы

1. Проведён анализ и объединение нескольких источников данных.
2. Выполнена очистка данных:
   - удалены дубликаты пользователей;
   - обработаны пропуски в категориальных признаках;
   - устранена мультиколлинеарность признаков.
3. Сформированы поведенческие признаки на основе логов посещений.
4. Построена базовая модель логистической регрессии.
5. Проведена балансировка классов и подбор гиперпараметров.
6. Подготовлен артефакт модели для внедрения.

### Итоговая модель

Лучшая модель:
- LogisticRegression (multinomial)
- class_weight="balanced"
- C = 5

Качество на тестовой выборке:
- Accuracy ≈ 0.56
- Macro F1 ≈ 0.52

Модель хорошо определяет крайние возрастные категории (младше 18 и 56+), однако испытывает сложности при разделении близких возрастных групп (например, 18–25 и 26–40 лет).

### Интерпретация результатов

Поведенческие признаки (время активности, глубина просмотра, рекламная активность, устройство) позволяют выявлять возрастные паттерны, однако различия между соседними возрастными группами выражены неярко, что ограничивает максимальное качество линейной модели.

### Ограничения

- Используются агрегированные признаки, без детальной поведенческой последовательности.
- Отсутствуют демографические или социально-экономические данные.
- Логистическая регрессия является линейной моделью и может не улавливать сложные нелинейные зависимости.

### Перспективы улучшения

- Расширение набора поведенческих признаков;
- Применение методов балансировки классов.

В целом модель демонстрирует стабильное качество и может быть использована как базовое решение для сегментации аудитории по возрастным категориям.